# DATA TRANSFORMATION FOR POWER BI
**Objective:** Read the master cleaned dataset and generate 4 specific aggregated tables for Power BI dashboards.

In [6]:
import pandas as pd
from pathlib import Path

# =====================================================================
# 1. LOAD MASTER DATA & SETUP EXPORT DIRECTORY
# =====================================================================
print("Loading cleaned master dataset...")

# Load the cleaned master dataset (Excel format)
df = pd.read_csv('../Data/clean_data/CovidDeaths_Cleaned.csv')

# Ensure the 'date' column is in the correct datetime format
df['date'] = pd.to_datetime(df['date'])

# Define and create the export directory if it doesn't exist to prevent OSError
export_path = Path('../Data/clean_data')
export_path.mkdir(parents=True, exist_ok=True)

Loading cleaned master dataset...


## 2. Generating Final Tables
Exporting 4 specific CSV files to streamline the Power BI import process.

In [7]:
# =========================================================
# TABLE 1: Global Numbers
# =========================================================
global_cases = df['new_cases'].sum()
global_deaths = df['new_deaths'].sum()

# Calculate global death percentage (handle division by zero safeguard)
global_death_pct = (global_deaths / global_cases * 100) if global_cases > 0 else 0

df_global = pd.DataFrame({
    'Total_Cases': [global_cases],
    'Total_Deaths': [global_deaths],
    'DeathPercentage': [global_death_pct]
})
df_global.to_csv(export_path / '1_GlobalNumbers.csv', index=False)
print("Exported Table 1: Global Numbers")


# =========================================================
# TABLE 2: Total Deaths Per Continent
# =========================================================
# Group by 'continent' since the master file is already cleaned 
# (invalid regions like 'World' or 'High income' are already removed)
df_deaths_continent = df.groupby('continent')['new_deaths'].sum().reset_index()

# Rename columns to match the required Power BI structure
df_deaths_continent.rename(columns={'new_deaths': 'TotalDeathCount'}, inplace=True)

# Sort values descending for better visualization in bar charts
df_deaths_continent.sort_values(by='TotalDeathCount', ascending=False, inplace=True)
df_deaths_continent.to_csv(export_path / '2_TotalDeathsPerContinent.csv', index=False)
print("Exported Table 2: Total Deaths Per Continent")


# =========================================================
# TABLE 3: Percent Population Infected (By Country)
# =========================================================
# Exclude rows with 0 or missing population to prevent calculation errors
df_valid_pop = df[df['population'] > 0].copy()

# Find the maximum cases for each country
df_infected = df_valid_pop.groupby(['location', 'population'])['total_cases'].max().reset_index()
df_infected['PercentPopulationInfected'] = (df_infected['total_cases'] / df_infected['population']) * 100

# Fill any missing percentage values with 0
df_infected['PercentPopulationInfected'] = df_infected['PercentPopulationInfected'].fillna(0)

# Sort by highest infection rate
df_infected.sort_values(by='PercentPopulationInfected', ascending=False, inplace=True)
df_infected.to_csv(export_path / '3_PercentPopulationInfected.csv', index=False)
print("Exported Table 3: Percent Population Infected")


# =========================================================
# TABLE 4: Percent Population Infected Over Time
# =========================================================
# Group by location, population, and date to get a time-series view
df_time_series = df_valid_pop.groupby(['location', 'population', 'date'])['total_cases'].max().reset_index()
df_time_series['PercentPopulationInfected'] = (df_time_series['total_cases'] / df_time_series['population']) * 100
df_time_series['PercentPopulationInfected'] = df_time_series['PercentPopulationInfected'].fillna(0)

# Sort chronologically to ensure the line chart in Power BI flows correctly
df_time_series.sort_values(by=['location', 'date'], inplace=True)
df_time_series.to_csv(export_path / '4_PercentPopulationInfected_TimeSeries.csv', index=False)
print("Exported Table 4: Percent Population Infected Over Time")

print("\nSuccess! All 4 tables have been successfully exported to the clean_data folder.")

Exported Table 1: Global Numbers
Exported Table 2: Total Deaths Per Continent
Exported Table 3: Percent Population Infected
Exported Table 4: Percent Population Infected Over Time

Success! All 4 tables have been successfully exported to the clean_data folder.
